# Quedas e aumentos bruscos de matrículas (INEP 2011–2025)

**Ideia do notebook:** para cada escola, calcular a variação percentual de matrículas **em relação ao ano anterior**, e usar boxplots + cerca de Tukey (Q1 − k·AIQ, Q3 + k·AIQ) para definir o que é uma queda (abaixo da cerca inferior) ou um aumento (acima da cerca superior) "bruscos".

**Como usar:**
1. Edite **apenas a célula "PARÂMETROS"** (etapa 1).
2. Rode *Run All*. No fim, abra o arquivo **`relatorio_quedas_bruscas.html`** (etapa 11): ele junta **todos os gráficos e tabelas** de todos os cenários (ex.: k=1,5 e k=2) em uma única página.
3. Para acumular mais cenários (outros k, quartis, critérios, etapas, filtros), use a seção **10) CENÁRIOS** da célula de parâmetros.

| Etapa | O que faz |
|---|---|
| 0 | Imports |
| 1 | **PARÂMETROS (único lugar para editar)** |
| 2 | Carrega a base e monta a variação % ano a ano |
| 3 | Calcula limites (por ano e global) e marca as quedas bruscas |
| 4 | Boxplot por ano (limite por ano × limite global) |
| 5 | Proporção de quedas e aumentos bruscos por ano |
| 6 | Cruzamento com rede, localização, região, UF e modalidade |
| 6.2b | **Boxplots lado a lado em cada ano (rural × urbana, rede, região...)** |
| 7 | Sensibilidade: vários quartis × vários multiplicadores |
| 8 | Comparação entre etapas (AI, AF, EM, total) |
| 9 | Exportação |
| 10 | **Cenários extras** (vários k / quartis / critérios / filtros) + comparação entre cenários |
| 11 | **Relatório HTML** com todos os gráficos |

## Etapa 0 — Imports

In [137]:
import io
import json
import time
import zipfile
import datetime
import html as _html
from contextlib import contextmanager
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()   # evita o limite de 5000 linhas do Altair


def estilizar(chart):
    """Visual limpo e consistente para todos os gráficos."""
    return (chart
            .configure_view(stroke=None)
            .configure_axis(gridColor="#E5E7EB", domainColor="#9CA3AF", tickColor="#9CA3AF",
                            labelFontSize=12, titleFontSize=13, labelColor="#374151", titleColor="#111827")
            .configure_title(fontSize=17, subtitleFontSize=12, subtitleColor="#6B7280", anchor="start", color="#111827")
            .configure_legend(labelFontSize=12, titleFontSize=12, symbolSize=120)
            .configure_range(category={"scheme": "set2"}))


pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

# ============================================================
#  COLETOR DO RELATÓRIO HTML
#  Tudo que passa por mostrar() / mostrar_tabela() / mostrar_texto()
#  aparece no notebook E é guardado para o relatório HTML (etapa 11).
# ============================================================
try:
    display
except NameError:                       # rodando fora do Jupyter
    def display(x):
        print(x)

RELATORIO = []                  # itens guardados para o relatório
CENARIO_ATUAL = "Principal"     # rótulo do cenário em execução (muda sozinho na etapa 10)
SECAO_ATUAL = "Geral"


def _exibir():
    return globals().get("EXIBIR_NO_NOTEBOOK", True)


def secao(nome):
    """Define a seção (título) sob a qual os próximos itens entram no relatório."""
    global SECAO_ATUAL
    SECAO_ATUAL = nome


def mostrar(chart, titulo, nota=None):
    """Mostra o gráfico no notebook e o guarda no relatório."""
    if chart is None:
        return
    try:
        spec = chart.to_dict()
    except Exception as e:
        spec = None
        print(f"⚠ Não consegui guardar '{titulo}' no relatório: {e}")
    if spec is not None:
        RELATORIO.append({"tipo": "grafico", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                          "titulo": titulo, "nota": nota, "spec": spec})
    if _exibir():
        display(chart)


def mostrar_tabela(df, titulo, nota=None, exibir=True):
    """Mostra a tabela no notebook e a guarda no relatório."""
    if df is None:
        return
    if isinstance(df, pd.Series):
        df = df.to_frame()
    tabela = df.to_html(classes="tabela", border=0, na_rep="–", float_format=lambda x: f"{x:,.2f}")
    RELATORIO.append({"tipo": "tabela", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                      "titulo": titulo, "nota": nota, "html": tabela})
    if exibir and _exibir():
        display(df)


def mostrar_texto(texto, titulo=None, nivel="info"):
    """Mensagem curta (nivel = 'info' ou 'aviso'): vai para o notebook e para o relatório."""
    RELATORIO.append({"tipo": "texto", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                      "titulo": titulo, "nota": None, "texto": texto, "nivel": nivel})
    if _exibir():
        print(("⚠ " if nivel == "aviso" else "") + texto)


def nome_cenario(cfg=None):
    """Rótulo legível do cenário a partir dos parâmetros em vigor."""
    nome = f"k={MULTIPLICADOR:g} · quartis {Q_INFERIOR:.2f}/{Q_SUPERIOR:.2f} · {CRITERIO}"
    fixos = ("MULTIPLICADOR", "Q_INFERIOR", "Q_SUPERIOR", "CRITERIO", "nome")
    extras = [f"{k}={v}" for k, v in (cfg or {}).items() if k not in fixos]
    return nome + (" · " + ", ".join(extras) if extras else "")


PARAMS_RESUMO = ["ETAPA", "INCLUIR_INTEGRAL", "ANO_INICIO", "ANO_FIM", "FILTRO_DEP_ADM", "FILTRO_LOCALIZACAO",
                 "FILTRO_REGIAO", "FILTRO_UF", "MATRICULA_MINIMA_ANO_ANTERIOR", "EXCLUIR_FECHAMENTO",
                 "Q_INFERIOR", "Q_SUPERIOR", "MULTIPLICADOR", "CRITERIO", "CORTE_ABSOLUTO_PCT",
                 "CORTE_ABSOLUTO_AUMENTO_PCT", "DIRECAO", "LIMITE_PARA_CRUZAMENTO", "ZOOM_Y", "ZOOM_MARGEM",
                 "Y_MIN", "Y_MAX", "ANOS_FOCO"]


def registrar_parametros():
    g = globals()
    tab = pd.DataFrame({"valor": [str(g.get(k, "—")) for k in PARAMS_RESUMO]}, index=PARAMS_RESUMO)
    tab.index.name = "parâmetro"
    mostrar_tabela(tab, "Parâmetros em vigor neste cenário", exibir=False)

## Etapa 1 — PARÂMETROS (edite só aqui)

In [138]:
# ============================================================
#  PARÂMETROS — TUDO QUE VOCÊ PRECISA EDITAR ESTÁ NESTA CÉLULA
# ============================================================

# ---------- 1) ARQUIVO ----------
CAMINHO_ARQUIVO = "bd5.zip"  # .zip (com um .csv/.xlsx/.parquet dentro), ou direto .csv, .xlsx, .parquet
NOME_ARQUIVO_NO_ZIP = None   # se o zip tiver vários arquivos, escreva o nome do que quer ler. None = usa o primeiro
CSV_SEP         = ";"        # se der erro de colunas, tente ";"
CSV_ENCODING    = "utf-8"    # se der erro de acentos, tente "latin-1"

# ---------- 2) QUAL "NÚMERO DE MATRÍCULAS" ANALISAR ----------
ETAPA            = "EM"   # "TOTAL" (AI+AF+EM), "AI", "AF" ou "EM"
INCLUIR_INTEGRAL = True      # True = regular + integral | False = só regular

# ---------- 3) FILTROS DA AMOSTRA (None = não filtra) ----------
ANO_INICIO   = 2012          # primeiro ano com variação (2011 não tem ano anterior)
ANO_FIM      = 2025
FILTRO_DEP_ADM     = None    # ex.: ["Privada"] ou ["Estadual", "Municipal"]
FILTRO_LOCALIZACAO = None    # ex.: ["Urbana"]
FILTRO_REGIAO      = None    # ex.: ["Sudeste", "Sul"]
FILTRO_UF          = None    # ex.: ["RJ", "SP"]

MATRICULA_MINIMA_ANO_ANTERIOR = 30    # ignora escolas muito pequenas (ruído). Use 0 para não filtrar
EXCLUIR_FECHAMENTO = True    # True = tira escolas com 0 matrículas no ano atual (queda de -100%)

# ---------- 4) CRITÉRIO DE "QUEDA BRUSCA" ----------
Q_INFERIOR    = 0.10         # quartil de baixo (0.25 = Q1). Ex.: 0.10 para P10
Q_SUPERIOR    = 0.90         # quartil de cima  (0.75 = Q3). Ex.: 0.90 para P90
MULTIPLICADOR = 1.5       # o "1,5" do 1,5·AIQ. Teste 1.0, 2.0, 2.5, 3.0...

CRITERIO = "estatistico"     # "estatistico" -> abaixo de Q_inf − k·AIQ
                             # "absoluto"    -> variação <= CORTE_ABSOLUTO_PCT
                             # "ambos"       -> precisa cumprir os dois
CORTE_ABSOLUTO_PCT = -30     # QUEDA: variação <= este valor (em %), usado se CRITERIO = "absoluto" ou "ambos"
CORTE_ABSOLUTO_AUMENTO_PCT = 30   # AUMENTO: variação >= este valor (em %), idem

# O mesmo CRITERIO vale para os dois lados:
#   estatistico -> queda abaixo de Q_inf − k·AIQ | aumento acima de Q_sup + k·AIQ

DIRECAO = "queda"            # o que analisar nas etapas 4, 6, 7, 8 e 9:
                             #   "queda"   -> só quedas bruscas
                             #   "aumento" -> só aumentos bruscos
                             #   "ambos"   -> qualquer variação brusca (queda OU aumento)
LIMITE_PARA_CRUZAMENTO = "ANO"   # qual flag usar nas etapas 6 e 9: "ANO" ou "GLOBAL"

# ---------- 5) GRÁFICOS ----------
ZOOM_Y = "meio"              # eixo y dos boxplots:
                             #   "meio"   -> aproxima da região central (caixas e medianas); cercas e outliers extremos saem da vista
                             #   "cercas" -> mostra também as cercas de Tukey (limites vermelho/laranja)
ZOOM_MARGEM = 0.5            # folga em volta (fração da amplitude das caixas). Menor = mais zoom. Ex.: 0.2 (muito perto) a 1.5 (mais folgado)
Y_MIN, Y_MAX = None, None    # opcional: fixa o eixo y manualmente (ignora ZOOM_Y). Ex.: Y_MIN, Y_MAX = -8, 8
LARGURA_GRAFICO = 1000       # largura dos gráficos (px)
ALTURA_GRAFICO  = 760        # altura dos gráficos (px)
COR_CAIXA       = "#5B8DB8"  # caixa dos anos normais
COR_CAIXA_FOCO  = "#8E6BBF"  # caixa dos ANOS_FOCO (destaca a pandemia)
COR_QUEDA       = "#D1495B"  # limite inferior / quedas
COR_AUMENTO     = "#E8A33D"  # limite superior / aumentos
ANOS_FOCO    = [2019, 2020, 2021, 2022]     # anos para comparar pandemia
MOSTRAR_PONTOS = "marcados"  # pontos no boxplot: "marcados" (conforme DIRECAO), "outliers" (fora da cerca, dos dois lados), "nenhum"
MAX_PONTOS_POR_ANO = 300     # amostra de pontos por ano (evita gráfico pesado)
BOXPLOT_POR = ["DEP_ADM", "LOCALIZACAO", "REGIAO"]   # dimensões para boxplot facetado
BOXPLOT_SO_ANOS_FOCO = True  # True = facetas só com ANOS_FOCO | False = todos os anos
N_MIN_CATEGORIA = 30         # categorias com menos escolas/ano que isso são escondidas nas proporções
BOXPLOT_ANO_ORIENTACAO = "horizontal"   # "horizontal" (anos na vertical) ou "vertical" (formato antigo)
BOXPLOT_ANO_ESCALA     = (-100, 100)    # eixo da variação %. None = usa o zoom de ZOOM_Y / ZOOM_MARGEM
BOXPLOT_ANO_LARGURA    = 1000           # largura (px)
BOXPLOT_ANO_ALTURA     = 1500           # altura (px): maior = mais comprido

# ---------- 6) CRUZAMENTOS ----------
# Dimensões usadas nas proporções. A modalidade predominante da etapa (ENS_PRED_AI/AF/EM)
# é adicionada automaticamente quando ETAPA != "TOTAL".
DIMENSOES = ["DEP_ADM", "LOCALIZACAO", "REGIAO", "UF"]
PERFIL_ANOS = ANOS_FOCO      # anos usados na tabela de perfil das escolas com queda

# ---------- 7) SENSIBILIDADE (grade de testes) ----------
GRADE_MULTIPLICADORES = [1.0, 1.5, 2.0, 2.5, 3.0]
GRADE_QUANTIS = [(0.25, 0.75), (0.10, 0.90), (0.05, 0.95)]   # pares (inferior, superior)

# ---------- 8) COMPARAÇÃO ENTRE ETAPAS ----------
ETAPAS_COMPARAR = ["TOTAL", "AI", "AF", "EM"]

# ---------- 9) EXPORTAÇÃO ----------
EXPORTAR_CSV = False
ARQUIVO_SAIDA = "escolas_queda_brusca.csv"

# ---------- 10) CENÁRIOS: acumular parâmetros no MESMO relatório (etapa 10) ----------
# O cenário "Principal" é o configurado nas seções acima (ex.: MULTIPLICADOR = 1.5).
# Os cenários abaixo rodam depois e entram no mesmo relatório + uma aba de comparação.
# Cada combinação de k × quartis × critério vira um cenário (o que for igual ao Principal é pulado).
CENARIO_MULTIPLICADORES = [1.5, 2.0]        # ex.: [1.5, 2.0, 3.0]. Lista vazia [] = não cria cenários
CENARIO_QUANTIS = [(0.25, 0.75)]            # pares (inferior, superior). Ex.: [(0.25, 0.75), (0.10, 0.90)]
CENARIO_CRITERIOS = ["estatistico"]         # ex.: ["estatistico", "absoluto", "ambos"]

# Cenários livres: qualquer parâmetro desta célula pode ser trocado. "nome" é opcional.
# Ex.: [{"nome": "Só ensino médio", "ETAPA": "EM"},
#       {"nome": "Sudeste, k=2", "FILTRO_REGIAO": ["Sudeste"], "MULTIPLICADOR": 2.0},
#       {"nome": "Só quedas, corte -40%", "DIRECAO": "queda", "CRITERIO": "absoluto", "CORTE_ABSOLUTO_PCT": -40}]
CENARIOS_EXTRA = []
CENARIOS_RODAR_ETAPAS = False   # True = refaz também a etapa 8 (comparação AI/AF/EM/total) em cada cenário (mais lento)

# ---------- 11) COMPARATIVOS: boxplots lado a lado, em cada ano (etapa 6.2b) ----------
COMPARATIVOS = ["LOCALIZACAO","DEP_ADM","REGIAO",""]  # dimensões comparadas. Ex.: ["LOCALIZACAO", "DEP_ADM", "REGIAO"]  (LOCALIZACAO = rural × urbana)
COMPARATIVO_FILTRO = None       # opcional: restringe categorias. Ex.: {"LOCALIZACAO": ["Rural", "Urbana"]}
COMPARATIVO_ANOS = None         # None = todos os anos. Ex.: [2019, 2020, 2021]
COMPARATIVO_LIMITES = ["geral", "categoria"]   # "geral"     -> cercas calculadas com TODAS as escolas do ano (mesmas das outras etapas)
                                               # "categoria" -> cada categoria (ex.: rural e urbana) com as SUAS cercas
                                               # com os dois, saem dois gráficos por dimensão

# ---------- 12) RELATÓRIO HTML (etapa 11) ----------
GERAR_RELATORIO = True
ARQUIVO_RELATORIO = "relatorio_quedas_bruscas.html"
TITULO_RELATORIO = "Quedas e aumentos bruscos de matrículas (INEP)"
EXIBIR_NO_NOTEBOOK = True       # False = não desenha nada no notebook (só monta o relatório; roda mais leve)
RELATORIO_ESTATICO = False      # False = gráficos interativos (tooltips; precisa de internet para carregar o Vega)
                                # True  = gráficos viram SVG fixos, arquivo 100% offline (precisa: pip install vl-convert-python)

## Etapa 2 — Carregar a base e calcular a variação % ano a ano

Para cada escola: `variação % = (matrículas no ano t − matrículas no ano t−1) / matrículas no ano t−1 × 100`.
Só entra a comparação se o ano anterior for **exatamente t−1** (escola que some e volta não gera comparação falsa).

In [139]:
def _ler(origem, ext):
    """Lê CSV/Excel/Parquet a partir de um caminho ou de bytes em memória."""
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(origem)
    if ext == ".parquet":
        return pd.read_parquet(origem)
    return pd.read_csv(origem, sep=CSV_SEP, encoding=CSV_ENCODING, low_memory=False)


def carregar(caminho):
    p = Path(caminho)
    if p.suffix.lower() != ".zip":
        return _ler(p, p.suffix.lower())

    with zipfile.ZipFile(p) as z:
        validos = [n for n in z.namelist()
                   if not n.endswith("/") and not n.startswith("__MACOSX")
                   and Path(n).suffix.lower() in (".csv", ".txt", ".xlsx", ".xls", ".parquet")]
        validos.sort(key=lambda n: Path(n).suffix.lower() == ".txt")   # .csv/.xlsx/.parquet têm prioridade sobre .txt
        print("Arquivos no zip:", validos)
        if not validos:
            raise FileNotFoundError("Nenhum .csv/.xlsx/.parquet encontrado dentro do zip.")
        nome = NOME_ARQUIVO_NO_ZIP or validos[0]
        if nome not in z.namelist():
            raise FileNotFoundError(f"'{nome}' não está no zip. Opções: {validos}")
        if NOME_ARQUIVO_NO_ZIP is None and len(validos) > 1:
            print(f"Aviso: o zip tem {len(validos)} arquivos; lendo '{nome}'. Use NOME_ARQUIVO_NO_ZIP para escolher.")
        print("Lendo:", nome)
        dados_bytes = io.BytesIO(z.read(nome))
    ext = Path(nome).suffix.lower()
    return _ler(dados_bytes, ".csv" if ext == ".txt" else ext)


def colunas_matricula(etapa, integral):
    etapas = ["AI", "AF", "EM"] if etapa == "TOTAL" else [etapa]
    cols = []
    for e in etapas:
        cols.append(f"QT_MAT_{e}_REG")
        if integral:
            cols.append(f"QT_MAT_{e}_INT")
    return cols


def preparar_base(raw, etapa, integral):
    """Devolve uma linha por escola/ano com MAT_ANT, MATRICULAS e VAR_PCT (já filtrada)."""
    cols_mat = colunas_matricula(etapa, integral)
    faltando = [c for c in ["ANO", "CO_ESCOLA"] + cols_mat if c not in raw.columns]
    if faltando:
        raise KeyError(f"Colunas ausentes na base: {faltando}")

    d = raw.copy()
    d["MATRICULAS"] = d[cols_mat].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

    # escola duplicada no mesmo ano -> soma as matrículas, mantém o resto
    if d.duplicated(["CO_ESCOLA", "ANO"]).any():
        n = d.duplicated(["CO_ESCOLA", "ANO"]).sum()
        print(f"Aviso: {n} linhas duplicadas (escola, ano). Somando matrículas.")
        outras = {c: "first" for c in d.columns if c not in ("CO_ESCOLA", "ANO", "MATRICULAS")}
        d = d.groupby(["CO_ESCOLA", "ANO"], as_index=False).agg({**outras, "MATRICULAS": "sum"})

    d = d.sort_values(["CO_ESCOLA", "ANO"]).reset_index(drop=True)
    g = d.groupby("CO_ESCOLA")
    d["ANO_ANT"] = g["ANO"].shift(1)
    d["MAT_ANT"] = g["MATRICULAS"].shift(1)

    d = d[(d["ANO"] - d["ANO_ANT"]) == 1]                    # só anos consecutivos
    d = d[d["MAT_ANT"] >= max(MATRICULA_MINIMA_ANO_ANTERIOR, 1)]   # evita divisão por zero
    if EXCLUIR_FECHAMENTO:
        d = d[d["MATRICULAS"] > 0]

    d = d[(d["ANO"] >= ANO_INICIO) & (d["ANO"] <= ANO_FIM)]
    for col, filtro in [("DEP_ADM", FILTRO_DEP_ADM), ("LOCALIZACAO", FILTRO_LOCALIZACAO),
                        ("REGIAO", FILTRO_REGIAO), ("UF", FILTRO_UF)]:
        if filtro is not None and col in d.columns:
            d = d[d[col].isin(filtro)]

    d["VAR_PCT"] = (d["MATRICULAS"] - d["MAT_ANT"]) / d["MAT_ANT"] * 100
    return d.reset_index(drop=True)


_CACHE_BASE = {}


def _chave_base(etapa):
    return repr((etapa, INCLUIR_INTEGRAL, ANO_INICIO, ANO_FIM, FILTRO_DEP_ADM, FILTRO_LOCALIZACAO,
                 FILTRO_REGIAO, FILTRO_UF, MATRICULA_MINIMA_ANO_ANTERIOR, EXCLUIR_FECHAMENTO))


def obter_base(etapa):
    """preparar_base com cache: cenários que só mudam k/quartis não refazem a base."""
    chave = _chave_base(etapa)
    if chave not in _CACHE_BASE:
        if len(_CACHE_BASE) >= 6:
            _CACHE_BASE.pop(next(iter(_CACHE_BASE)))
        _CACHE_BASE[chave] = preparar_base(raw, etapa, INCLUIR_INTEGRAL)
    return _CACHE_BASE[chave]


raw = carregar(CAMINHO_ARQUIVO)
print("Base bruta:", raw.shape, "| anos:", raw["ANO"].min(), "a", raw["ANO"].max())

RELATORIO.clear()        # a cada execução a partir daqui, o relatório recomeça do zero
_CACHE_BASE.clear()
NOME_PRINCIPAL = "Principal · " + nome_cenario()
CENARIO_ATUAL = NOME_PRINCIPAL
secao("Parâmetros")
registrar_parametros()
secao("Etapa 2 — Base e variação % ano a ano")

base = obter_base(ETAPA)
print("Observações escola-ano usadas:", len(base))
print("Escolas distintas:", base["CO_ESCOLA"].nunique())

DIMS = [d for d in DIMENSOES if d in base.columns]
if ETAPA != "TOTAL" and f"ENS_PRED_{ETAPA}" in base.columns:
    DIMS.append(f"ENS_PRED_{ETAPA}")
print("Dimensões de cruzamento:", DIMS)

desc = base.groupby("ANO")["VAR_PCT"].describe(percentiles=[.05, .25, .5, .75, .95])
mostrar_tabela(desc, "Distribuição da variação % por ano")

Arquivos no zip: ['bd5.csv']
Lendo: bd5.csv
Base bruta: (2051631, 27) | anos: 2011 a 2025
Observações escola-ano usadas: 370684
Escolas distintas: 32955
Dimensões de cruzamento: ['DEP_ADM', 'LOCALIZACAO', 'REGIAO', 'UF', 'ENS_PRED_EM']


,count,mean,std,min,5%,25%,50%,75%,95%,max
ANO,,,,,,,,,,
2012,"25,124.00",3.28,28.92,-97.04,-24.31,-8.49,0.00,9.56,39.53,"1,121.59"
2013,"25,474.00",1.49,26.14,-94.87,-25.68,-9.30,-0.93,8.32,34.26,"1,055.93"
2014,"25,767.00",1.91,22.59,-94.12,-24.42,-8.53,0.00,8.96,33.33,614.81
2015,"26,079.00",-0.69,23.62,-99.78,-27.34,-11.25,-2.49,6.51,29.63,968.75
2016,"26,301.00",1.51,21.92,-97.50,-26.23,-9.31,0.00,9.46,33.33,334.48
2017,"26,527.00",-1.02,25.04,-91.40,-28.21,-11.86,-2.86,6.07,30.19,"1,280.52"
2018,"26,663.00",-0.96,23.22,-96.55,-28.92,-12.05,-2.71,6.74,31.75,664.47
2019,"26,708.00",-1.42,25.16,-98.68,-28.81,-12.43,-3.12,6.15,29.79,"1,141.42"
2020,"26,714.00",2.10,22.33,-98.05,-25.81,-8.12,0.75,10.14,32.76,912.24


### Diagnóstico: onde as linhas se perdem em cada ano?
Se só aparecer 1 ano nos gráficos, esta tabela mostra o motivo:
- `linhas_brutas`: linhas do arquivo naquele ano;
- `com_matricula>0`: linhas com matrícula positiva na etapa escolhida;
- `com_ano_anterior`: escolas que também existem no ano t−1 (mesmo `CO_ESCOLA`);
- `usadas_na_analise`: o que sobrou depois de todos os filtros.

In [140]:
chaves = raw[["CO_ESCOLA", "ANO"]].drop_duplicates()
anteriores = chaves.assign(ANO=chaves["ANO"] + 1)
com_prev = chaves.merge(anteriores, on=["CO_ESCOLA", "ANO"]).groupby("ANO").size()

mat_raw = raw[colunas_matricula(ETAPA, INCLUIR_INTEGRAL)].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

diag = pd.DataFrame({
    "linhas_brutas": raw.groupby("ANO").size(),
    "com_matricula>0": raw[mat_raw > 0].groupby("ANO").size(),
    "com_ano_anterior": com_prev,
    "usadas_na_analise": base.groupby("ANO").size(),
}).fillna(0).astype(int)
mostrar_tabela(diag, "Diagnóstico: onde as linhas se perdem em cada ano")

if base["ANO"].nunique() < 2:
    mostrar_texto("Só 1 ano chegou na análise. Veja a tabela acima: se 'linhas_brutas' tem vários anos mas "
                  "'com_ano_anterior' é 0, o CO_ESCOLA não bate entre anos; se 'com_matricula>0' é 0, "
                  "as colunas QT_MAT_* estão vazias nesses anos.", nivel="aviso")

,linhas_brutas,com_matricula>0,com_ano_anterior,usadas_na_analise
ANO,,,,
2011,151509,26944,0,0
2012,150305,27165,146436,25124
2013,147185,27455,143863,25474
2014,143958,27748,141316,25767
2015,140864,27997,138526,26079
2016,139813,28334,137116,26301
2017,137296,28543,134637,26527
2018,134454,28650,132086,26663
2019,132456,28837,130056,26708


## Etapa 3 — Limites de Tukey e marcação das quedas bruscas

- **Limite por ano:** Q1/Q3 calculados dentro de cada ano → "queda atípica *naquele ano*".
- **Limite global:** Q1/Q3 com todos os anos juntos → um único corte. Mostra o efeito de anos como 2020/2021.

Colunas criadas (True/False): `QUEDA_ANO`, `QUEDA_GLOBAL`, `AUMENTO_ANO`, `AUMENTO_GLOBAL` e `BRUSCA_ANO`, `BRUSCA_GLOBAL` (queda OU aumento).

In [141]:
def _atualizar_derivadas():
    """Recalcula as variáveis que dependem dos parâmetros (também usada pelos cenários da etapa 10)."""
    global PREFIXO, ROTULO, FLAG, DIMS
    PREFIXO = {"queda": "QUEDA", "aumento": "AUMENTO", "ambos": "BRUSCA"}[DIRECAO]
    ROTULO = {"queda": "queda brusca", "aumento": "aumento brusco",
              "ambos": "variação brusca (queda ou aumento)"}[DIRECAO]
    FLAG = f"{PREFIXO}_{LIMITE_PARA_CRUZAMENTO}"
    dims = [d for d in DIMENSOES if d in raw.columns]
    if ETAPA != "TOTAL" and f"ENS_PRED_{ETAPA}" in raw.columns:
        dims.append(f"ENS_PRED_{ETAPA}")
    DIMS = dims


_atualizar_derivadas()


def aplicar_limites(d, q_inf, q_sup, mult, criterio, corte_abs):
    d = d.copy()

    # --- por ano ---
    tab = d.groupby("ANO")["VAR_PCT"].agg(
        N="size",
        Q_INF=lambda s: s.quantile(q_inf),
        MEDIANA="median",
        Q_SUP=lambda s: s.quantile(q_sup),
    )
    tab["AIQ"] = tab["Q_SUP"] - tab["Q_INF"]
    tab["LIM_INF_ANO"] = tab["Q_INF"] - mult * tab["AIQ"]
    tab["LIM_SUP_ANO"] = tab["Q_SUP"] + mult * tab["AIQ"]
    d = d.join(tab[["LIM_INF_ANO", "LIM_SUP_ANO"]], on="ANO")

    # --- global ---
    s = d["VAR_PCT"]
    q1, q3 = s.quantile(q_inf), s.quantile(q_sup)
    aiq = q3 - q1
    glob = {"Q_INF": q1, "MEDIANA": s.median(), "Q_SUP": q3, "AIQ": aiq,
            "LIM_INF": q1 - mult * aiq, "LIM_SUP": q3 + mult * aiq}
    d["LIM_INF_GLOBAL"] = glob["LIM_INF"]
    d["LIM_SUP_GLOBAL"] = glob["LIM_SUP"]

    # --- flags ---
    if criterio not in ("estatistico", "absoluto", "ambos"):
        raise ValueError("CRITERIO deve ser 'estatistico', 'absoluto' ou 'ambos'")
    abs_q = d["VAR_PCT"] <= corte_abs
    abs_a = d["VAR_PCT"] >= CORTE_ABSOLUTO_AUMENTO_PCT
    for modo in ["ANO", "GLOBAL"]:
        est_q = d["VAR_PCT"] < d[f"LIM_INF_{modo}"]
        est_a = d["VAR_PCT"] > d[f"LIM_SUP_{modo}"]
        if criterio == "estatistico":
            q, a = est_q, est_a
        elif criterio == "absoluto":
            q, a = abs_q, abs_a
        else:
            q, a = est_q & abs_q, est_a & abs_a
        d[f"QUEDA_{modo}"] = q
        d[f"AUMENTO_{modo}"] = a
        d[f"BRUSCA_{modo}"] = q | a
    return d, tab, glob


def relatar_limites(tab, glob):
    mostrar_texto("Limites GLOBAIS: " + " | ".join(f"{k} = {v:.2f}" for k, v in glob.items()), "Limites globais")
    mostrar_tabela(tab.round(2), "Limites por ano (quartis, AIQ e cercas)")
    inalc = tab.index[tab["LIM_INF_ANO"] <= -100].tolist()
    if glob["LIM_INF"] <= -100:
        mostrar_texto("Limite inferior GLOBAL <= -100%: nenhuma escola pode ser outlier negativo por esse critério.",
                      nivel="aviso")
    if inalc:
        mostrar_texto(f"Anos com limite inferior <= -100% (inalcançável pelo critério estatístico): {inalc}. "
                      "Dica: use CRITERIO='absoluto' ou diminua MULTIPLICADOR.", nivel="aviso")


dados, tab_ano, lim_global = aplicar_limites(
    base, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)

secao("Etapa 3 — Limites de Tukey e marcação")
relatar_limites(tab_ano, lim_global)

Limites GLOBAIS: Q_INF = -20.16 | MEDIANA = -0.98 | Q_SUP = 21.92 | AIQ = 42.07 | LIM_INF = -83.26 | LIM_SUP = 85.03


,N,Q_INF,MEDIANA,Q_SUP,AIQ,LIM_INF_ANO,LIM_SUP_ANO
ANO,,,,,,,
2012,25124,-17.26,0.00,24.24,41.50,-79.51,86.50
2013,25474,-18.33,-0.93,21.21,39.54,-77.65,80.52
2014,25767,-17.47,0.00,21.35,38.83,-75.71,79.60
2015,26079,-20.39,-2.49,18.39,38.78,-78.57,76.57
2016,26301,-19.08,0.00,21.68,40.76,-80.21,82.82
2017,26527,-21.15,-2.86,18.28,39.43,-80.29,77.43
2018,26663,-21.84,-2.71,19.38,41.22,-83.68,81.21
2019,26708,-21.81,-3.12,18.41,40.22,-82.13,78.74
2020,26714,-17.98,0.75,21.91,39.89,-77.81,81.74


## Etapa 4 — Boxplots por ano

- Caixa = `Q_INFERIOR` a `Q_SUPERIOR` (com 0.25/0.75 é o boxplot clássico). Traço branco = mediana.
- Bigodes = menor/maior valor **dentro** da cerca. Traço **vermelho** = limite inferior (abaixo = queda brusca). Traço **laranja** = limite superior (acima = aumento brusco).
- Pontos = conforme `MOSTRAR_PONTOS`.
- O eixo y é ampliado na região central (`ZOOM_Y`, `ZOOM_MARGEM`). Para ver as cercas e os outliers, use `ZOOM_Y = "cercas"`.

In [142]:
def estatisticas_box(d, grupo, modo):
    li, ls, flag = f"LIM_INF_{modo}", f"LIM_SUP_{modo}", f"{PREFIXO}_{modo}"
    linhas = []
    for chave, g in d.groupby(grupo):
        chave = chave if isinstance(chave, tuple) else (chave,)
        v = g["VAR_PCT"]
        dentro = v[(v >= g[li].iloc[0]) & (v <= g[ls].iloc[0])]
        linha = dict(zip(grupo, chave))
        linha.update({
            "q_inf": v.quantile(Q_INFERIOR), "mediana": v.median(), "q_sup": v.quantile(Q_SUPERIOR),
            "bigode_inf": dentro.min(), "bigode_sup": dentro.max(),
            "lim_inf": g[li].iloc[0], "lim_sup": g[ls].iloc[0], "n": len(v), "pct_marcada": g[flag].mean() * 100,
        })
        linhas.append(linha)
    return pd.DataFrame(linhas)


def dominio_y(d, stats=None):
    """Eixo y: fixo (Y_MIN/Y_MAX), zoom no 'meio' (caixas) ou nas 'cercas'."""
    if Y_MIN is not None and Y_MAX is not None:
        return [float(Y_MIN), float(Y_MAX)]
    if stats is not None:                                   # facetas: usa as caixas de cada categoria
        q_lo, q_hi = stats["q_inf"].min(), stats["q_sup"].max()
    else:
        q_lo = d.groupby("ANO")["VAR_PCT"].quantile(Q_INFERIOR).min()
        q_hi = d.groupby("ANO")["VAR_PCT"].quantile(Q_SUPERIOR).max()
    if ZOOM_Y == "cercas":
        lo = min(q_lo, d["LIM_INF_ANO"].min(), d["LIM_INF_GLOBAL"].min())
        hi = max(q_hi, d["LIM_SUP_ANO"].max(), d["LIM_SUP_GLOBAL"].max())
        pad = 0.10 * (hi - lo)
    else:
        lo, hi = q_lo, q_hi
        pad = ZOOM_MARGEM * max(hi - lo, 1e-6)
    return [float(max(-100, lo - pad)), float(hi + pad)]


def tamanho_caixa(n_x, largura):
    return int(max(8, min(70, largura / max(n_x, 1) * 0.5)))


def camadas_boxplot(stats, base_chart, x_enc, dom, tam, cor_por=None):
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    foco = alt.FieldOneOfPredicate(field="ANO", oneOf=[int(a) for a in ANOS_FOCO])
    if cor_por:   # facetas: uma cor por categoria
        cor_caixa = alt.Color(f"{cor_por}:N", legend=None, scale=alt.Scale(scheme="set2"))
    else:
        cor_caixa = alt.condition(foco, alt.value(COR_CAIXA_FOCO), alt.value(COR_CAIXA))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]
    zero = base_chart.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(y=alt.Y(datum=0, scale=escala))
    bigode = base_chart.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        x_enc, y=alt.Y("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), y2="bigode_sup:Q")
    tampa_i = base_chart.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(x_enc, y=alt.Y("bigode_inf:Q", scale=escala))
    tampa_s = base_chart.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(x_enc, y=alt.Y("bigode_sup:Q", scale=escala))
    caixa = base_chart.mark_bar(size=tam, cornerRadius=3, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        x_enc, y=alt.Y("q_inf:Q", scale=escala), y2="q_sup:Q", color=cor_caixa, tooltip=tip)
    mediana = base_chart.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(x_enc, y=alt.Y("mediana:Q", scale=escala))
    lim_i = base_chart.mark_tick(color=COR_QUEDA, size=tam * 1.4, thickness=3, clip=True).encode(x_enc, y=alt.Y("lim_inf:Q", scale=escala))
    lim_s = base_chart.mark_tick(color=COR_AUMENTO, size=tam * 1.4, thickness=3, clip=True).encode(x_enc, y=alt.Y("lim_sup:Q", scale=escala))
    return [zero, bigode, tampa_i, tampa_s, caixa, mediana, lim_i, lim_s]


LEGENDA_BOX = ("Caixa = quartis {qi:.0%}–{qs:.0%} · traço branco = mediana · bigodes = dados dentro da cerca · "
               "traço vermelho = limite inferior · traço laranja = limite superior · roxo = anos em foco (gráfico geral) / cor = categoria (facetas)")


def pontos_boxplot(d, modo):
    flag, li, ls = f"{PREFIXO}_{modo}", f"LIM_INF_{modo}", f"LIM_SUP_{modo}"
    if MOSTRAR_PONTOS == "nenhum":
        return None
    if MOSTRAR_PONTOS == "marcados":
        p = d[d[flag]]
    else:
        p = d[(d["VAR_PCT"] < d[li]) | (d["VAR_PCT"] > d[ls])]
    p = p.sample(frac=1, random_state=0).groupby("ANO").head(MAX_PONTOS_POR_ANO)
    cols = [c for c in ["ANO", "VAR_PCT", "CO_ESCOLA", "NO_ESCOLA", "DEP_ADM", "UF", "MAT_ANT", "MATRICULAS"] if c in p.columns]
    return p[cols]



def camadas_boxplot_horizontal(stats, dom, tam):
    """Boxplot deitado: anos no eixo y (de cima para baixo) e variação % no eixo x."""
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    y_enc = alt.Y("ANO:O", title="Ano")
    foco = alt.FieldOneOfPredicate(field="ANO", oneOf=[int(a) for a in ANOS_FOCO])
    cor_caixa = alt.condition(foco, alt.value(COR_CAIXA_FOCO), alt.value(COR_CAIXA))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]
    b = alt.Chart(stats)
    zero = b.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(x=alt.X(datum=0, scale=escala))
    bigode = b.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        y_enc, x=alt.X("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), x2="bigode_sup:Q")
    tampa_i = b.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(y_enc, x=alt.X("bigode_inf:Q", scale=escala))
    tampa_s = b.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(y_enc, x=alt.X("bigode_sup:Q", scale=escala))
    caixa = b.mark_bar(size=tam, cornerRadius=3, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        y_enc, x=alt.X("q_inf:Q", scale=escala), x2="q_sup:Q", color=cor_caixa, tooltip=tip)
    mediana = b.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(y_enc, x=alt.X("mediana:Q", scale=escala))
    lim_i = b.mark_tick(color=COR_QUEDA, size=tam * 1.4, thickness=3, clip=True).encode(y_enc, x=alt.X("lim_inf:Q", scale=escala))
    lim_s = b.mark_tick(color=COR_AUMENTO, size=tam * 1.4, thickness=3, clip=True).encode(y_enc, x=alt.X("lim_sup:Q", scale=escala))
    return [zero, bigode, tampa_i, tampa_s, caixa, mediana, lim_i, lim_s]


def boxplot_por_ano(d, modo, titulo):
    stats = estatisticas_box(d, ["ANO"], modo)
    horizontal = BOXPLOT_ANO_ORIENTACAO == "horizontal"
    if BOXPLOT_ANO_ESCALA is not None:
        dom = [float(BOXPLOT_ANO_ESCALA[0]), float(BOXPLOT_ANO_ESCALA[1])]
        txt_eixo = f"Eixo fixo: {dom[0]:.0f}% a {dom[1]:.0f}% (cercas = traços vermelho e laranja; o que passa do eixo fica fora da vista)"
    else:
        dom = dominio_y(d)
        txt_eixo = (f"Eixo ampliado: {dom[0]:.1f}% a {dom[1]:.1f}% (o que sai dessa faixa fica fora da vista; "
                    "os números continuam nas tabelas e tooltips)")
    larg, altu = (BOXPLOT_ANO_LARGURA, BOXPLOT_ANO_ALTURA) if horizontal else (LARGURA_GRAFICO, ALTURA_GRAFICO)
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    pts = pontos_boxplot(d, modo)
    cor_pts = alt.condition(alt.datum.VAR_PCT > 0, alt.value(COR_AUMENTO), alt.value(COR_QUEDA))
    jitter = "sqrt(-2*log(random()))*cos(2*PI*random())"

    if horizontal:
        tam = tamanho_caixa(len(stats), altu)
        camadas = camadas_boxplot_horizontal(stats, dom, tam)
        if pts is not None and len(pts):
            camadas.append(
                alt.Chart(pts).mark_circle(size=26, opacity=0.5, clip=True).encode(
                    y=alt.Y("ANO:O", title="Ano"), yOffset="jitter:Q",
                    x=alt.X("VAR_PCT:Q", scale=escala), color=cor_pts, tooltip=[c for c in pts.columns])
                .transform_calculate(jitter=jitter))
    else:
        tam = tamanho_caixa(len(stats), larg)
        x_enc = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
        camadas = camadas_boxplot(stats, alt.Chart(stats), x_enc, dom, tam)
        if pts is not None and len(pts):
            camadas.append(
                alt.Chart(pts).mark_circle(size=26, opacity=0.5, clip=True).encode(
                    x_enc, xOffset="jitter:Q", y=alt.Y("VAR_PCT:Q", scale=escala),
                    color=cor_pts, tooltip=[c for c in pts.columns])
                .transform_calculate(jitter=jitter))

    graf = alt.layer(*camadas).properties(
        width=larg, height=altu,
        title=alt.Title(titulo, subtitle=[LEGENDA_BOX.format(qi=Q_INFERIOR, qs=Q_SUPERIOR), txt_eixo]))
    return estilizar(graf)

def rotulo_params():
    return (f"quartis {Q_INFERIOR:.2f}/{Q_SUPERIOR:.2f}, k={MULTIPLICADOR}, critério={CRITERIO}, "
            f"etapa={ETAPA}, mín. {MATRICULA_MINIMA_ANO_ANTERIOR} alunos")


sub = rotulo_params()

secao("Etapa 4 — Boxplots por ano")
mostrar(boxplot_por_ano(dados, "ANO", f"Limite POR ANO — {sub}"),
        "Boxplot por ano — limite calculado dentro de cada ano")

alt.LayerChart(...)

In [143]:
mostrar(boxplot_por_ano(dados, "GLOBAL", f"Limite GLOBAL — {sub}"),
        "Boxplot por ano — limite global (todos os anos juntos)")

alt.LayerChart(...)

## Etapa 5 — Quantas escolas tiveram queda ou aumento brusco em cada ano?
O boxplot mostra a distribuição; esta etapa resume **proporção** (e contagem) de escolas marcadas.

In [144]:
def resumo_por_ano(d):
    """Tabela (contagem e %) e gráfico de escolas com queda/aumento brusco por ano."""
    resumo = pd.DataFrame({"escolas": d.groupby("ANO").size()})
    for tipo in ["QUEDA", "AUMENTO"]:
        for modo in ["ANO", "GLOBAL"]:
            resumo[f"n_{tipo}_{modo}"] = d.groupby("ANO")[f"{tipo}_{modo}"].sum()
            resumo[f"pct_{tipo}_{modo}"] = resumo[f"n_{tipo}_{modo}"] / resumo["escolas"] * 100

    pcts = [c for c in resumo.columns if c.startswith("pct_")]
    longo = resumo.reset_index().melt(id_vars="ANO", value_vars=pcts, var_name="serie", value_name="pct")
    longo["tipo"] = np.where(longo["serie"].str.contains("QUEDA"), "queda brusca", "aumento brusco")
    longo["limite"] = np.where(longo["serie"].str.endswith("_ANO"), "limite por ano", "limite global")

    graf = estilizar(alt.Chart(longo).mark_line(point=alt.OverlayMarkDef(size=90, filled=True), strokeWidth=3).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("pct:Q", title="% de escolas"),
        color=alt.Color("tipo:N", title=None, scale=alt.Scale(domain=["queda brusca", "aumento brusco"], range=[COR_QUEDA, COR_AUMENTO])),
        strokeDash=alt.StrokeDash("limite:N", title=None),
        tooltip=["ANO:O", "tipo:N", "limite:N", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.8), title="Proporção de escolas com queda e aumento brusco por ano"))
    return resumo, graf


secao("Etapa 5 — Proporção de quedas e aumentos por ano")
resumo, graf_resumo = resumo_por_ano(dados)
mostrar_tabela(resumo.round(2), "Contagem e % de escolas marcadas por ano")
mostrar(graf_resumo, "Proporção de quedas e aumentos bruscos por ano")

,escolas,n_QUEDA_ANO,pct_QUEDA_ANO,n_QUEDA_GLOBAL,pct_QUEDA_GLOBAL,n_AUMENTO_ANO,pct_AUMENTO_ANO,n_AUMENTO_GLOBAL,pct_AUMENTO_GLOBAL
ANO,,,,,,,,,
2012,25124,19,0.08,12,0.05,342,1.36,357,1.42
2013,25474,25,0.10,11,0.04,265,1.04,244,0.96
2014,25767,43,0.17,18,0.07,277,1.08,247,0.96
2015,26079,21,0.08,15,0.06,254,0.97,200,0.77
2016,26301,18,0.07,13,0.05,246,0.94,233,0.89
2017,26527,20,0.08,11,0.04,277,1.04,228,0.86
2018,26663,8,0.03,8,0.03,254,0.95,234,0.88
2019,26708,17,0.06,16,0.06,234,0.88,193,0.72
2020,26714,25,0.09,15,0.06,200,0.75,183,0.69


alt.Chart(...)

## Etapa 6 — Cruzamento com rede, localização, região, UF e modalidade

Usa a flag escolhida em `LIMITE_PARA_CRUZAMENTO`. Sempre compara **proporção** dentro de cada categoria (não contagem).

In [145]:
FLAG = f"{PREFIXO}_{LIMITE_PARA_CRUZAMENTO}"
print("Flag usada nos cruzamentos:", FLAG, "->", ROTULO)


def proporcao_por(d, dim, flag):
    t = d.groupby(["ANO", dim]).agg(N=(flag, "size"), QUEDAS=(flag, "sum")).reset_index()
    t["PCT_QUEDA"] = t["QUEDAS"] / t["N"] * 100
    return t[t["N"] >= N_MIN_CATEGORIA]


def grafico_proporcao(t, dim):
    n_cat = t[dim].nunique()
    tip = ["ANO:O", f"{dim}:N", "N:Q", "QUEDAS:Q", alt.Tooltip("PCT_QUEDA:Q", format=".2f")]
    if n_cat <= 8:
        g = alt.Chart(t).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
            x=alt.X("ANO:O", title="Ano"), y=alt.Y("PCT_QUEDA:Q", title=f"% com {ROTULO}"),
            color=alt.Color(f"{dim}:N"), tooltip=tip).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.7))
    else:   # muitas categorias (ex.: UF) -> mapa de calor
        g = alt.Chart(t).mark_rect().encode(
            x=alt.X("ANO:O", title="Ano"), y=alt.Y(f"{dim}:N", title=dim),
            color=alt.Color("PCT_QUEDA:Q", scale=alt.Scale(scheme="orangered"), title="% marcadas"),
            tooltip=tip).properties(width=LARGURA_GRAFICO, height=max(300, 20 * n_cat))
    return estilizar(g.properties(title=f"% de escolas com {ROTULO} por {dim} (limite {LIMITE_PARA_CRUZAMENTO.lower()})"))


def relatar_cruzamentos(d):
    for dim in DIMS:
        t = proporcao_por(d, dim, FLAG)
        if t.empty:
            mostrar_texto(f"[{dim}] nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano.", nivel="aviso")
            continue
        mostrar(grafico_proporcao(t, dim), f"% de escolas com {ROTULO} por {dim}")


secao("Etapa 6 — Cruzamentos")
relatar_cruzamentos(dados)

Flag usada nos cruzamentos: QUEDA_ANO -> queda brusca


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### 6.1 — Tabela de pandemia: % de escolas marcadas (conforme `DIRECAO`) por rede nos anos-foco

In [146]:
def relatar_pandemia(d):
    if "DEP_ADM" in d.columns:
        tp = (d[d["ANO"].isin(ANOS_FOCO)]
              .groupby(["DEP_ADM", "ANO"])[FLAG].mean().mul(100).unstack("ANO"))
        mostrar_tabela(tp.round(2), f"% de escolas marcadas ({ROTULO}) por rede nos anos {ANOS_FOCO}")


secao("Etapa 6.1 — Tabela de pandemia")
relatar_pandemia(dados)

ANO,2019,2020,2021,2022
DEP_ADM,,,,
Estadual,0.05,0.10,0.03,0.00
Federal,0.00,0.00,0.00,0.00
Municipal,0.62,0.00,0.00,0.00
Privada,0.08,0.10,0.03,0.01


### 6.2 — Boxplots facetados por categoria
Mesmos limites da etapa 3 (calculados com todas as escolas do ano). Sem pontos individuais, para ficar leve.

In [147]:
def boxplot_facetado(d, dim, modo):
    dd = d[d["ANO"].isin(ANOS_FOCO)] if BOXPLOT_SO_ANOS_FOCO else d
    if dd.empty:
        print(f"[{dim}] Nenhuma linha nos anos {ANOS_FOCO}. Anos disponíveis: {sorted(int(a) for a in d['ANO'].unique())}")
        return None
    dd = dd.copy()
    dd[dim] = dd[dim].fillna("(vazio)").astype(str)   # NaN não derruba o groupby
    stats = estatisticas_box(dd, ["ANO", dim], modo)
    if stats.empty or "n" not in stats.columns:
        print(f"[{dim}] Sem dados para o boxplot.")
        return None
    stats = stats[stats["n"] >= N_MIN_CATEGORIA]
    if stats.empty:
        print(f"[{dim}] Nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano. Diminua N_MIN_CATEGORIA.")
        return None
    n_fac = stats[dim].nunique()
    n_anos = stats["ANO"].nunique()
    larg = int(max(110, (LARGURA_GRAFICO - 60 * n_fac) / n_fac))
    tam = tamanho_caixa(n_anos, larg)
    x_enc = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
    dom = dominio_y(d, stats)
    camadas = camadas_boxplot(stats, alt.Chart(stats), x_enc, dom, tam, cor_por=dim)
    graf = (alt.layer(*camadas).properties(width=larg, height=int(ALTURA_GRAFICO * 0.85))
            .facet(column=alt.Column(f"{dim}:N", title=None, header=alt.Header(labelFontSize=14, labelFontWeight="bold")))
            .properties(title=alt.Title(f"Boxplot por {dim} (limite {modo.lower()})",
                                        subtitle=[LEGENDA_BOX.format(qi=Q_INFERIOR, qs=Q_SUPERIOR), f"Eixo y ampliado: {dom[0]:.1f}% a {dom[1]:.1f}%"])))
    return estilizar(graf)


def relatar_facetados(d):
    for dim in BOXPLOT_POR:
        if dim in d.columns:
            graf = boxplot_facetado(d, dim, LIMITE_PARA_CRUZAMENTO)
            mostrar(graf, f"Boxplot facetado por {dim}")


secao("Etapa 6.2 — Boxplots facetados")
relatar_facetados(dados)

alt.FacetChart(...)

alt.FacetChart(...)

alt.FacetChart(...)

### 6.2b — Boxplots lado a lado em cada ano (ex.: rural × urbana)

Em cada ano, uma caixa por categoria da dimensão escolhida em `COMPARATIVOS` (por padrão `LOCALIZACAO` = rural × urbana).
- `COMPARATIVO_LIMITES = ["geral", "categoria"]` gera **dois gráficos**: um com as cercas de Tukey do conjunto de todas as escolas do ano (mesmo critério das outras etapas) e outro em que cada categoria tem **as suas próprias cercas**.
- Dá para comparar também `DEP_ADM` e `REGIAO`, restringir categorias (`COMPARATIVO_FILTRO`) e anos (`COMPARATIVO_ANOS`).
- Junto de cada gráfico saem duas tabelas: **mediana** e **% de escolas marcadas** por ano e categoria.

In [148]:
def _sem_marcas(d):
    """Remove colunas de limites/flags para poder recalcular com aplicar_limites()."""
    return d.drop(columns=[c for c in d.columns
                           if c.startswith(("LIM_INF_", "LIM_SUP_", "QUEDA_", "AUMENTO_", "BRUSCA_"))])


def boxplot_comparativo(d, dim, modo, limites="geral", anos=None, categorias=None):
    """Boxplots lado a lado: em cada ano, uma caixa por categoria de `dim`. Devolve (gráfico, estatísticas)."""
    dd = d.copy()
    dd[dim] = dd[dim].fillna("(vazio)").astype(str)
    if categorias:
        dd = dd[dd[dim].isin([str(c) for c in categorias])]
    if dd.empty:
        mostrar_texto(f"[{dim}] nenhuma linha para as categorias pedidas.", nivel="aviso")
        return None, None
    if limites == "categoria":      # cercas próprias de cada categoria
        dd = pd.concat([aplicar_limites(_sem_marcas(g), Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO,
                                        CORTE_ABSOLUTO_PCT)[0] for _, g in dd.groupby(dim)])
    if anos:
        dd = dd[dd["ANO"].isin([int(a) for a in anos])]
    if dd.empty:
        mostrar_texto(f"[{dim}] nenhuma linha nos anos pedidos.", nivel="aviso")
        return None, None

    stats = estatisticas_box(dd, ["ANO", dim], modo)
    stats = stats[stats["n"] >= N_MIN_CATEGORIA].copy()
    if stats.empty:
        mostrar_texto(f"[{dim}] nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano. "
                      "Diminua N_MIN_CATEGORIA.", nivel="aviso")
        return None, None

    cats = sorted(stats[dim].unique())
    n_cat, n_anos = len(cats), stats["ANO"].nunique()
    dom = dominio_y(dd, stats)
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    tam = int(max(4, min(60, LARGURA_GRAFICO / max(n_anos, 1) * 0.8 / n_cat)))

    x = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
    off = alt.XOffset(f"{dim}:N", sort=cats)
    cor = alt.Color(f"{dim}:N", title=dim, scale=alt.Scale(domain=cats, scheme="set2"))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip(f"{dim}:N", title=dim), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]

    b = alt.Chart(stats)
    zero = b.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(y=alt.Y(datum=0, scale=escala))
    bigode = b.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        x, xOffset=off, y=alt.Y("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), y2="bigode_sup:Q")
    caixa = b.mark_bar(size=tam, cornerRadius=2, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        x, xOffset=off, y=alt.Y("q_inf:Q", scale=escala), y2="q_sup:Q", color=cor, tooltip=tip)
    mediana = b.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(
        x, xOffset=off, y=alt.Y("mediana:Q", scale=escala))
    lim_i = b.mark_tick(color=COR_QUEDA, size=tam, thickness=2.5, clip=True).encode(
        x, xOffset=off, y=alt.Y("lim_inf:Q", scale=escala))
    lim_s = b.mark_tick(color=COR_AUMENTO, size=tam, thickness=2.5, clip=True).encode(
        x, xOffset=off, y=alt.Y("lim_sup:Q", scale=escala))

    rot = "cercas gerais (todas as escolas do ano)" if limites == "geral" else "cercas próprias de cada categoria"
    legenda = (f"Caixa = quartis {Q_INFERIOR:.0%}–{Q_SUPERIOR:.0%} · traço branco = mediana · bigodes = dados dentro da cerca · "
               "traço vermelho/laranja = cerca inferior/superior · uma cor por categoria")
    graf = alt.layer(zero, bigode, caixa, mediana, lim_i, lim_s).properties(
        width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.85),
        title=alt.Title(f"Boxplot de {dim} em cada ano — {rot}",
                        subtitle=[legenda, f"{rotulo_params()} · eixo y ampliado: {dom[0]:.1f}% a {dom[1]:.1f}%"]))
    return estilizar(graf), stats


def relatar_comparativos(d):
    modo = LIMITE_PARA_CRUZAMENTO
    for dim in COMPARATIVOS:
        if dim not in d.columns:
            mostrar_texto(f"Comparativo ignorado: coluna {dim} não existe na base.", nivel="aviso")
            continue
        filtro = (COMPARATIVO_FILTRO or {}).get(dim)
        for i, lim in enumerate(COMPARATIVO_LIMITES):
            graf, stats = boxplot_comparativo(d, dim, modo, lim, COMPARATIVO_ANOS, filtro)
            if graf is None:
                continue
            rot = "cercas gerais" if lim == "geral" else "cercas próprias de cada categoria"
            mostrar(graf, f"{dim}: boxplots lado a lado em cada ano ({rot})")
            if i == 0:
                mostrar_tabela(stats.pivot(index="ANO", columns=dim, values="mediana").round(2),
                               f"{dim}: mediana da variação % por ano")
            mostrar_tabela(stats.pivot(index="ANO", columns=dim, values="pct_marcada").round(2),
                           f"{dim}: % de escolas marcadas por ano ({rot})")


secao("Etapa 6.2b — Comparativos lado a lado (rural × urbana etc.)")
relatar_comparativos(dados)

alt.LayerChart(...)

LOCALIZACAO,Rural,Urbana
ANO,,
2012,0.00,-0.05
2013,0.00,-0.99
2014,1.41,0.00
2015,-0.78,-2.61
2016,0.99,-0.23
2017,-1.18,-2.98
2018,-1.05,-2.84
2019,-3.88,-3.06
2020,0.39,0.78


LOCALIZACAO,Rural,Urbana
ANO,,
2012,0.10,0.07
2013,0.14,0.09
2014,0.31,0.15
2015,0.17,0.07
2016,0.08,0.07
2017,0.12,0.07
2018,0.12,0.02
2019,0.12,0.06
2020,0.12,0.09


alt.LayerChart(...)

LOCALIZACAO,Rural,Urbana
ANO,,
2012,0.00,0.10
2013,0.05,0.10
2014,0.04,0.17
2015,0.00,0.08
2016,0.00,0.08
2017,0.00,0.07
2018,0.00,0.02
2019,0.00,0.06
2020,0.08,0.10


...

REGIAO,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
ANO,,,,,
2012,0.37,-1.01,1.32,0.00,-0.31
2013,-0.90,-1.38,0.24,-1.12,0.00
2014,-0.09,-1.27,0.84,0.00,0.43
2015,-2.56,-2.58,-0.65,-2.20,-3.86
2016,-3.46,0.65,-1.31,1.95,-5.36
2017,-4.73,-1.42,-1.07,-3.01,-4.48
2018,1.10,-1.61,-1.40,-4.57,-1.37
2019,0.00,-3.05,-0.64,-4.72,-1.94
2020,2.74,0.56,0.59,0.00,3.12


REGIAO,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
ANO,,,,,
2012,0.05,0.11,0.17,0.06,0.02
2013,0.21,0.17,0.27,0.04,0.00
2014,0.10,0.19,0.32,0.20,0.02
2015,0.00,0.11,0.21,0.07,0.05
2016,0.05,0.11,0.15,0.04,0.05
2017,0.10,0.14,0.25,0.03,0.02
2018,0.05,0.05,0.14,0.01,0.00
2019,0.10,0.06,0.14,0.07,0.00
2020,0.14,0.05,0.14,0.12,0.05


alt.LayerChart(...)

REGIAO,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
ANO,,,,,
2012,0.00,0.02,0.06,0.15,0.05
2013,0.10,0.02,0.16,0.05,0.00
2014,0.00,0.02,0.16,0.23,0.17
2015,0.00,0.08,0.16,0.11,0.02
2016,0.05,0.03,0.05,0.08,0.07
2017,0.00,0.09,0.00,0.05,0.02
2018,0.05,0.02,0.00,0.07,0.00
2019,0.10,0.05,0.00,0.09,0.00
2020,0.10,0.03,0.14,0.13,0.16


⚠ Comparativo ignorado: coluna  não existe na base.


In [149]:
ETAPAS_BOX = ["AI", "AF", "EM"]   # etapas lado a lado (pode incluir "TOTAL")


def relatar_etapas_lado_a_lado():
    partes = []
    for et in ETAPAS_BOX:
        try:
            b = obter_base(et)
        except KeyError as e:
            mostrar_texto(f"Etapa {et} ignorada: {e}", nivel="aviso")
            continue
        dd, _, _ = aplicar_limites(b, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)
        partes.append(dd.assign(ETAPA_ANALISADA=et))
    if not partes:
        return
    d = pd.concat(partes, ignore_index=True)
    graf, stats = boxplot_comparativo(d, "ETAPA_ANALISADA", LIMITE_PARA_CRUZAMENTO, "geral", COMPARATIVO_ANOS, None)
    if graf is None:
        return
    mostrar(graf, "Etapas lado a lado em cada ano (cercas de cada etapa)")
    mostrar_tabela(stats.pivot(index="ANO", columns="ETAPA_ANALISADA", values="mediana").round(2),
                   "Etapas: mediana da variação % por ano")
    mostrar_tabela(stats.pivot(index="ANO", columns="ETAPA_ANALISADA", values="pct_marcada").round(2),
                   "Etapas: % de escolas marcadas por ano")


secao("Etapa 6.2c — Etapas lado a lado")
relatar_etapas_lado_a_lado()

alt.LayerChart(...)

ETAPA_ANALISADA,AF,AI,EM
ANO,,,
2012,-2.35,-2.86,0.00
2013,-2.63,-2.22,-0.93
2014,-3.17,-1.06,0.00
2015,-2.57,-1.82,-2.49
2016,-1.27,-1.67,0.00
2017,-1.79,-1.27,-2.86
2018,-0.50,-1.56,-2.71
2019,-1.18,-1.69,-3.12
2020,0.00,-2.10,0.75


ETAPA_ANALISADA,AF,AI,EM
ANO,,,
2012,0.07,0.11,0.08
2013,0.13,0.15,0.10
2014,0.09,0.13,0.17
2015,0.08,0.17,0.08
2016,0.13,0.19,0.07
2017,0.11,0.20,0.08
2018,0.14,0.21,0.03
2019,0.14,0.21,0.06
2020,0.18,0.20,0.09


### 6.3 — Perfil das escolas marcadas
Compara a composição das escolas **marcadas (conforme `DIRECAO`)** com a composição de **todas** as escolas (nos anos de `PERFIL_ANOS`).
`razao > 1` → a categoria está sobrerrepresentada entre as marcadas.

In [150]:
def relatar_perfil(d):
    sub_p = d[d["ANO"].isin(PERFIL_ANOS)]
    if sub_p.empty:
        mostrar_texto(f"Nenhuma linha nos anos {PERFIL_ANOS} para montar o perfil.", nivel="aviso")
        return
    for dim in DIMS:
        comp = pd.DataFrame({
            "% das escolas (todas)": sub_p[dim].value_counts(normalize=True) * 100,
            "% das escolas marcadas": sub_p.loc[sub_p[FLAG], dim].value_counts(normalize=True) * 100,
            "n_marcadas": sub_p.loc[sub_p[FLAG], dim].value_counts(),
        }).fillna(0)
        comp["razao"] = comp["% das escolas marcadas"] / comp["% das escolas (todas)"]
        mostrar_tabela(comp.sort_values("razao", ascending=False).round(2), f"Perfil por {dim} (anos {PERFIL_ANOS})")


secao("Etapa 6.3 — Perfil das escolas marcadas")
relatar_perfil(dados)

,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
DEP_ADM,,,,
Municipal,0.60,2.00,1.00,3.35
Privada,26.59,32.00,16.00,1.20
Estadual,70.63,66.00,33.00,0.93
Federal,2.18,0.00,0.00,0.00


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
LOCALIZACAO,,,,
Rural,9.56,14.00,7,1.46
Urbana,90.44,86.00,43,0.95


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
REGIAO,,,,
Centro-Oeste,7.91,14.00,7,1.77
Norte,8.09,14.00,7,1.73
Sudeste,43.30,52.00,26,1.20
Nordeste,24.71,14.00,7,0.57
Sul,16.00,6.00,3,0.37


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
UF,,,,
AC,0.59,6.00,3.00,10.23
MS,1.46,6.00,3.00,4.11
AM,1.67,4.00,2.00,2.40
DF,0.85,2.00,1.00,2.36
SP,22.62,38.00,19.00,1.68
BA,5.39,8.00,4.00,1.48
PA,2.99,4.00,2.00,1.34
ES,1.52,2.00,1.00,1.31
GO,3.41,4.00,2.00,1.17


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
ENS_PRED_EM,,,,
Integral,13.58,30.00,15.00,2.21
Regular,86.04,70.00,35.00,0.81
Misto,0.38,0.00,0.00,0.00


## Etapa 7 — Sensibilidade: quartis × multiplicadores
Refaz a classificação para todas as combinações de `GRADE_QUANTIS` × `GRADE_MULTIPLICADORES`
e mostra como a **% de escolas marcadas** (conforme `DIRECAO`) muda em cada ano.

In [151]:
def sensibilidade(d, grade_q, grade_m, criterio, corte):
    linhas = []
    for qi, qs in grade_q:
        for m in grade_m:
            dd, _, _ = aplicar_limites(d, qi, qs, m, criterio, corte)
            for modo in ["ANO", "GLOBAL"]:
                r = dd.groupby("ANO")[f"{PREFIXO}_{modo}"].mean() * 100
                for ano, val in r.items():
                    linhas.append({"quantis": f"{qi:.2f}/{qs:.2f}", "k": m,
                                   "limite": "por ano" if modo == "ANO" else "global",
                                   "ANO": ano, "pct": val})
    return pd.DataFrame(linhas)


sens = sensibilidade(base, GRADE_QUANTIS, GRADE_MULTIPLICADORES, CRITERIO, CORTE_ABSOLUTO_PCT)

graf_sens = estilizar(alt.Chart(sens).mark_line(point=True, strokeWidth=2.2).encode(
    x=alt.X("ANO:O", title="Ano"), y=alt.Y("pct:Q", title=f"% de escolas com {ROTULO}"),
    color=alt.Color("k:O", title="multiplicador k", scale=alt.Scale(scheme="viridis")),
    tooltip=["quantis", "k", "limite", "ANO:O", alt.Tooltip("pct:Q", format=".2f")]
).properties(width=int(LARGURA_GRAFICO / 2.4), height=220).facet(
    row=alt.Row("quantis:N", title="quartis (inf/sup)"), column=alt.Column("limite:N", title=None)
).properties(title="Sensibilidade do critério"))

secao("Etapa 7 — Sensibilidade")
mostrar(graf_sens, "Sensibilidade: % de escolas marcadas por quartis × k")

alt.FacetChart(...)

In [152]:
# Resumo: % médio de escolas marcadas (todos os anos) para cada combinação
resumo_sens = (sens.groupby(["limite", "quantis", "k"])["pct"].mean().unstack("k"))
mostrar_tabela(resumo_sens.round(2), "% médio de escolas marcadas (todos os anos) por quartis × k")

k                  1.00  1.50  2.00  2.50  3.00
limite  quantis                                
global  0.05/0.95  0.02  0.00  0.00  0.00  0.00
        0.10/0.90  0.39  0.05  0.00  0.00  0.00
        0.25/0.75  4.43  2.09  1.05  0.55  0.27
por ano 0.05/0.95  0.03  0.00  0.00  0.00  0.00
        0.10/0.90  0.39  0.06  0.00  0.00  0.00
        0.25/0.75  4.43  2.08  1.07  0.56  0.27

## Etapa 8 — Comparação entre etapas (AI, AF, EM, total)
Roda todo o pipeline para cada etapa de `ETAPAS_COMPARAR`, com os mesmos parâmetros das etapas anteriores.

In [153]:
def comparar_etapas():
    saida = []
    for et in ETAPAS_COMPARAR:
        try:
            b = obter_base(et)
            dd, _, _ = aplicar_limites(b, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)
            r = dd.groupby("ANO")[FLAG].mean().mul(100).rename("pct").reset_index()
            r["etapa"] = et
            saida.append(r)
        except KeyError as e:
            mostrar_texto(f"Etapa {et} ignorada: {e}", nivel="aviso")
    return pd.concat(saida) if saida else None


def relatar_etapas():
    comp = comparar_etapas()
    if comp is None:
        return
    mostrar(estilizar(alt.Chart(comp).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano"), y=alt.Y("pct:Q", title=f"% com {ROTULO}"),
        color="etapa:N", tooltip=["etapa", "ANO:O", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.7),
                 title=f"% de escolas com {ROTULO} por etapa (limite {LIMITE_PARA_CRUZAMENTO.lower()})")),
        "% de escolas marcadas por etapa")
    mostrar_tabela(comp.pivot(index="ANO", columns="etapa", values="pct").round(2), "% de escolas marcadas por etapa")


secao("Etapa 8 — Comparação entre etapas")
relatar_etapas()

alt.Chart(...)

etapa,AF,AI,EM,TOTAL
ANO,,,,
2012,0.07,0.11,0.08,0.19
2013,0.13,0.15,0.10,0.25
2014,0.09,0.13,0.17,0.25
2015,0.08,0.17,0.08,0.25
2016,0.13,0.19,0.07,0.24
2017,0.11,0.20,0.08,0.29
2018,0.14,0.21,0.03,0.34
2019,0.14,0.21,0.06,0.30
2020,0.18,0.20,0.09,0.28


## Etapa 9 — Exportação e lista das escolas

In [154]:
colunas_saida = [c for c in ["ANO", "CO_ESCOLA", "NO_ESCOLA", "UF", "NO_MUNICIPIO", "REGIAO", "LOCALIZACAO",
                             "DEP_ADM", "MAT_ANT", "MATRICULAS", "VAR_PCT", "QUEDA_ANO", "QUEDA_GLOBAL", "AUMENTO_ANO", "AUMENTO_GLOBAL"]
                 if c in dados.columns]


def lista_marcadas(d):
    cols = [c for c in colunas_saida if c in d.columns]
    return d.loc[d[FLAG], cols].sort_values("VAR_PCT", key=lambda x: -x.abs())


quedas = lista_marcadas(dados)
secao("Etapa 9 — Lista das escolas marcadas")
mostrar_texto(f"{len(quedas)} observações escola-ano com {ROTULO} ({FLAG}), ordenadas pela maior variação absoluta.")
mostrar_tabela(quedas.head(20), f"As 20 maiores variações ({FLAG})")

if EXPORTAR_CSV:
    quedas.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")
    print("Salvo em", ARQUIVO_SAIDA)

228 observações escola-ano com queda brusca (QUEDA_ANO), ordenadas pela maior variação absoluta.


,ANO,CO_ESCOLA,NO_ESCOLA,UF,NO_MUNICIPIO,REGIAO,LOCALIZACAO,DEP_ADM,MAT_ANT,MATRICULAS,VAR_PCT,QUEDA_ANO,QUEDA_GLOBAL,AUMENTO_ANO,AUMENTO_GLOBAL
183599,2015,33079595,COL REALENGO,RJ,Rio de Janeiro,Sudeste,Urbana,Privada,921.00,2,-99.78,True,True,False,False
244066,2021,35119039,COLEGIO CERMAC,SP,São Paulo,Sudeste,Urbana,Privada,90.00,1,-98.89,True,True,False,False
256530,2019,35170458,AUGUSTO HELOI COLEGIO,SP,São Paulo,Sudeste,Urbana,Privada,76.00,1,-98.68,True,True,False,False
119285,2023,29459052,CENTRO TECNOLOGICO DE APRENDIZAGEM SRA SANTANA...,BA,Alagoinhas,Nordeste,Urbana,Privada,267.00,4,-98.50,True,True,False,False
314553,2020,42059712,EEB RUI BARBOSA,SC,São Lourenço do Oeste,Sul,Urbana,Estadual,205.00,4,-98.05,True,True,False,False
365142,2021,52085465,FACULDADE SENAI ROBERTO MANGE,GO,Anápolis,Centro-Oeste,Urbana,Privada,196.00,4,-97.96,True,True,False,False
186245,2022,33092818,J LUDOLF REIS COLEGIO,RJ,São João de Meriti,Sudeste,Urbana,Privada,44.00,1,-97.73,True,True,False,False
160976,2016,31347671,GALILEU PRE VESTIBULAR,MG,Frutal,Sudeste,Urbana,Privada,40.00,1,-97.50,True,True,False,False
34942,2015,21155801,COL CAXIENSE,MA,Caxias,Nordeste,Urbana,Privada,36.00,1,-97.22,True,True,False,False
22782,2012,15584100,COLEGIO DE E F M LEONARDO DA VINCI,PA,Belém,Norte,Urbana,Privada,169.00,5,-97.04,True,True,False,False


## Etapa 10 — Cenários extras (acumular parâmetros no mesmo relatório)

Roda **todo o pipeline** (etapas 3 a 6, 9 e, se quiser, 8) para cada cenário definido na seção **10) CENÁRIOS** dos parâmetros:
- `CENARIO_MULTIPLICADORES` × `CENARIO_QUANTIS` × `CENARIO_CRITERIOS` → todas as combinações (ex.: k=1,5 e k=2);
- `CENARIOS_EXTRA` → cenários livres (outra etapa, outro filtro, outra direção, outro corte absoluto...).

Cada cenário vira uma **aba** no relatório HTML, e uma aba **"Comparação entre cenários"** mostra lado a lado a % de escolas marcadas e as cercas por ano.

In [155]:
def resumo_cenario(d):
    """% de escolas com queda e com aumento brusco por ano (para comparar cenários)."""
    linhas = []
    for tipo, rotulo in [("QUEDA", "queda brusca"), ("AUMENTO", "aumento brusco")]:
        r = d.groupby("ANO")[f"{tipo}_{LIMITE_PARA_CRUZAMENTO}"].mean().mul(100)
        linhas.append(r.rename("pct").reset_index().assign(tipo=rotulo))
    return pd.concat(linhas, ignore_index=True)


def limites_cenario(tab):
    t = tab[["LIM_INF_ANO", "LIM_SUP_ANO"]].reset_index().melt(id_vars="ANO", var_name="cerca", value_name="valor")
    t["cerca"] = t["cerca"].map({"LIM_INF_ANO": "cerca inferior", "LIM_SUP_ANO": "cerca superior"})
    return t


@contextmanager
def cenario_ativo(cfg, nome=None):
    """Troca temporariamente os parâmetros de `cfg` e restaura tudo no fim."""
    global CENARIO_ATUAL, SECAO_ATUAL
    antigos = {k: globals()[k] for k in cfg}
    ant_cen, ant_sec = CENARIO_ATUAL, SECAO_ATUAL
    try:
        globals().update(cfg)
        _atualizar_derivadas()
        nome = nome or nome_cenario(cfg)
        usados = {x["cenario"] for x in RELATORIO}
        n = 2
        base_nome = nome
        while nome in usados:
            nome = f"{base_nome} ({n})"
            n += 1
        CENARIO_ATUAL = nome
        yield nome
    finally:
        globals().update(antigos)
        _atualizar_derivadas()
        CENARIO_ATUAL, SECAO_ATUAL = ant_cen, ant_sec


def rodar_cenario(cfg_in):
    cfg = {k: v for k, v in cfg_in.items() if k != "nome"}
    desconhecidos = [k for k in cfg if k not in globals()]
    if desconhecidos:
        raise KeyError(f"Parâmetro(s) desconhecido(s) no cenário: {desconhecidos}")
    t0 = time.time()
    with cenario_ativo(cfg, cfg_in.get("nome")) as nome:
        print(f"▶ Cenário: {nome}")
        secao("Parâmetros")
        registrar_parametros()
        d, tab, glob = aplicar_limites(obter_base(ETAPA), Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)

        secao("Etapa 3 — Limites de Tukey e marcação")
        relatar_limites(tab, glob)

        secao("Etapa 4 — Boxplots por ano")
        s = rotulo_params()
        mostrar(boxplot_por_ano(d, "ANO", f"Limite POR ANO — {s}"), "Boxplot por ano — limite calculado dentro de cada ano")
        mostrar(boxplot_por_ano(d, "GLOBAL", f"Limite GLOBAL — {s}"), "Boxplot por ano — limite global (todos os anos juntos)")

        secao("Etapa 5 — Proporção de quedas e aumentos por ano")
        resumo_, graf_ = resumo_por_ano(d)
        mostrar_tabela(resumo_.round(2), "Contagem e % de escolas marcadas por ano")
        mostrar(graf_, "Proporção de quedas e aumentos bruscos por ano")

        secao("Etapa 6 — Cruzamentos")
        relatar_cruzamentos(d)
        secao("Etapa 6.1 — Tabela de pandemia")
        relatar_pandemia(d)
        secao("Etapa 6.2 — Boxplots facetados")
        relatar_facetados(d)
        secao("Etapa 6.2b — Comparativos lado a lado (rural × urbana etc.)")
        relatar_comparativos(d)
        secao("Etapa 6.3 — Perfil das escolas marcadas")
        relatar_perfil(d)
        if CENARIOS_RODAR_ETAPAS:
            secao("Etapa 8 — Comparação entre etapas")
            relatar_etapas()
        secao("Etapa 9 — Lista das escolas marcadas")
        mostrar_tabela(lista_marcadas(d).head(20), f"As 20 maiores variações ({FLAG})")
        print(f"   concluído em {time.time() - t0:.0f}s")
        return nome, resumo_cenario(d), limites_cenario(tab)


def montar_cenarios():
    def chave(c):
        if c["CRITERIO"] == "absoluto":     # no critério absoluto, k e quartis não mudam nada: evita cenários repetidos
            return ("absoluto",)
        return (c["Q_INFERIOR"], c["Q_SUPERIOR"], float(c["MULTIPLICADOR"]), c["CRITERIO"])

    vistos = {chave(dict(Q_INFERIOR=Q_INFERIOR, Q_SUPERIOR=Q_SUPERIOR, MULTIPLICADOR=MULTIPLICADOR, CRITERIO=CRITERIO))}
    lista = []
    for k, (qi, qs), crit in product(CENARIO_MULTIPLICADORES, CENARIO_QUANTIS, CENARIO_CRITERIOS):
        cfg = dict(MULTIPLICADOR=k, Q_INFERIOR=qi, Q_SUPERIOR=qs, CRITERIO=crit)
        if chave(cfg) not in vistos:        # o que for igual ao Principal já foi rodado
            vistos.add(chave(cfg))
            lista.append(cfg)
    lista.extend(dict(c) for c in CENARIOS_EXTRA)
    return lista


cenarios = montar_cenarios()
print(f"Principal + {len(cenarios)} cenário(s) extra(s).")

comp_res = [resumo_cenario(dados).assign(cenario=NOME_PRINCIPAL)]
comp_lim = [limites_cenario(tab_ano).assign(cenario=NOME_PRINCIPAL)]
for cfg in cenarios:
    nome_c, r_, l_ = rodar_cenario(cfg)
    comp_res.append(r_.assign(cenario=nome_c))
    comp_lim.append(l_.assign(cenario=nome_c))

if len(comp_res) > 1:
    CENARIO_ATUAL = "Comparação entre cenários"
    res = pd.concat(comp_res, ignore_index=True)
    lim = pd.concat(comp_lim, ignore_index=True)

    secao("% de escolas marcadas por ano")
    graf = (alt.Chart(res).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("pct:Q", title="% de escolas marcadas"),
        color=alt.Color("cenario:N", title="cenário", legend=alt.Legend(orient="bottom", columns=1)),
        tooltip=["cenario:N", "ANO:O", "tipo:N", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=int(LARGURA_GRAFICO / 1.8), height=int(ALTURA_GRAFICO * 0.6))
        .facet(column=alt.Column("tipo:N", title=None))
        .properties(title=f"% de escolas com queda e com aumento brusco, por cenário (limite {LIMITE_PARA_CRUZAMENTO.lower()})"))
    mostrar(estilizar(graf), "Quedas e aumentos bruscos por ano — todos os cenários")
    for tipo in ["queda brusca", "aumento brusco"]:
        piv = res[res["tipo"] == tipo].pivot(index="ANO", columns="cenario", values="pct")
        mostrar_tabela(piv.round(2), f"% de escolas com {tipo} por ano, por cenário")

    secao("Cercas de Tukey por ano")
    graf = (alt.Chart(lim).mark_line(point=alt.OverlayMarkDef(size=60, filled=True), strokeWidth=2.2).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("valor:Q", title="limite (variação %)"),
        color=alt.Color("cenario:N", title="cenário", legend=alt.Legend(orient="bottom", columns=1)),
        tooltip=["cenario:N", "ANO:O", "cerca:N", alt.Tooltip("valor:Q", format=".2f")]
    ).properties(width=int(LARGURA_GRAFICO / 1.8), height=int(ALTURA_GRAFICO * 0.6))
        .facet(column=alt.Column("cerca:N", title=None))
        .properties(title="Cercas de Tukey por ano, por cenário"))
    mostrar(estilizar(graf), "Cercas por ano — todos os cenários")
    for cerca in ["cerca inferior", "cerca superior"]:
        piv = lim[lim["cerca"] == cerca].pivot(index="ANO", columns="cenario", values="valor")
        mostrar_tabela(piv.round(2), f"{cerca.capitalize()} por ano, por cenário")

CENARIO_ATUAL = NOME_PRINCIPAL

Principal + 2 cenário(s) extra(s).
▶ Cenário: k=1.5 · quartis 0.25/0.75 · estatistico
Limites GLOBAIS: Q_INF = -10.20 | MEDIANA = -0.98 | Q_SUP = 8.79 | AIQ = 18.99 | LIM_INF = -38.69 | LIM_SUP = 37.27


,N,Q_INF,MEDIANA,Q_SUP,AIQ,LIM_INF_ANO,LIM_SUP_ANO
ANO,,,,,,,
2012,25124,-8.49,0.00,9.56,18.05,-35.57,36.64
2013,25474,-9.30,-0.93,8.32,17.62,-35.74,34.76
2014,25767,-8.53,0.00,8.96,17.50,-34.78,35.21
2015,26079,-11.25,-2.49,6.51,17.76,-37.89,33.15
2016,26301,-9.31,0.00,9.46,18.77,-37.47,37.62
2017,26527,-11.86,-2.86,6.07,17.93,-38.75,32.96
2018,26663,-12.05,-2.71,6.74,18.79,-40.24,34.93
2019,26708,-12.43,-3.12,6.15,18.58,-40.30,34.03
2020,26714,-8.12,0.75,10.14,18.26,-35.52,37.54


alt.LayerChart(...)

alt.LayerChart(...)

,escolas,n_QUEDA_ANO,pct_QUEDA_ANO,n_QUEDA_GLOBAL,pct_QUEDA_GLOBAL,n_AUMENTO_ANO,pct_AUMENTO_ANO,n_AUMENTO_GLOBAL,pct_AUMENTO_GLOBAL
ANO,,,,,,,,,
2012,25124,494,1.97,400,1.59,1391,5.54,1359,5.41
2013,25474,558,2.19,448,1.76,1240,4.87,1118,4.39
2014,25767,556,2.16,427,1.66,1199,4.65,1084,4.21
2015,26079,497,1.91,470,1.80,1081,4.15,886,3.40
2016,26301,499,1.90,455,1.73,1065,4.05,1087,4.13
2017,26527,553,2.08,556,2.10,1170,4.41,966,3.64
2018,26663,504,1.89,561,2.10,1133,4.25,1041,3.90
2019,26708,531,1.99,590,2.21,1077,4.03,930,3.48
2020,26714,631,2.36,505,1.89,1040,3.89,1050,3.93


...

cenario,Principal · k=1.5 · quartis 0.10/0.90 · estatistico,k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,,
2012,0.08,1.97,1.12
2013,0.10,2.19,1.17
2014,0.17,2.16,1.17
2015,0.08,1.91,0.98
2016,0.07,1.90,0.90
2017,0.08,2.08,1.11
2018,0.03,1.89,1.05
2019,0.06,1.99,1.01
2020,0.09,2.36,1.33


cenario,Principal · k=1.5 · quartis 0.10/0.90 · estatistico,k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,,
2012,1.36,5.54,4.08
2013,1.04,4.87,3.32
2014,1.08,4.65,3.15
2015,0.97,4.15,2.85
2016,0.94,4.05,2.68
2017,1.04,4.41,3.09
2018,0.95,4.25,2.91
2019,0.88,4.03,2.66
2020,0.75,3.89,2.47


alt.FacetChart(...)

cenario,Principal · k=1.5 · quartis 0.10/0.90 · estatistico,k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,,
2012,-79.51,-35.57,-44.59
2013,-77.65,-35.74,-44.55
2014,-75.71,-34.78,-43.53
2015,-78.57,-37.89,-46.78
2016,-80.21,-37.47,-46.86
2017,-80.29,-38.75,-47.72
2018,-83.68,-40.24,-49.63
2019,-82.13,-40.30,-49.59
2020,-77.81,-35.52,-44.65


cenario,Principal · k=1.5 · quartis 0.10/0.90 · estatistico,k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,,
2012,86.50,36.64,45.66
2013,80.52,34.76,43.57
2014,79.60,35.21,43.96
2015,76.57,33.15,42.03
2016,82.82,37.62,47.01
2017,77.43,32.96,41.93
2018,81.21,34.93,44.33
2019,78.74,34.03,43.32
2020,81.74,37.54,46.67


## Etapa 11 — Relatório HTML

Junta **todos os gráficos e tabelas** registrados ao longo do notebook (cenário principal + cenários extras + comparação) em um único arquivo HTML, com uma aba por cenário, índice por seção e seções que recolhem.
- Gráficos interativos (tooltip, botão de exportar PNG/SVG). Precisam de internet só para carregar o Vega; para um arquivo 100% offline use `RELATORIO_ESTATICO = True`.
- O arquivo é salvo na pasta do notebook (`ARQUIVO_RELATORIO`).

In [156]:
_CSS = """
:root{--bg:#F8FAFC;--card:#FFFFFF;--tx:#111827;--sub:#6B7280;--bd:#E5E7EB;--ac:#2F5D8A;--av:#FEF3C7;--avb:#F59E0B}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.5 -apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
header{background:#0F2A43;color:#fff;padding:28px 32px}
header h1{margin:0 0 6px;font-size:26px}
header p{margin:0;color:#CBD5E1;font-size:13px}
.abas{position:sticky;top:0;z-index:10;display:flex;gap:6px;flex-wrap:wrap;padding:10px 32px;background:#fff;border-bottom:1px solid var(--bd)}
.aba{border:1px solid var(--bd);background:#fff;color:var(--tx);border-radius:999px;padding:6px 14px;cursor:pointer;font-size:13px}
.aba:hover{border-color:var(--ac)}
.aba.ativa{background:var(--ac);border-color:var(--ac);color:#fff}
main{padding:20px 32px 60px;max-width:1400px;margin:0 auto}
.cenario>h2.cen{font-size:20px;margin:26px 0 8px;color:var(--ac);border-bottom:2px solid var(--ac);padding-bottom:6px}
.indice{display:flex;gap:8px;flex-wrap:wrap;margin:8px 0 14px}
.indice a{font-size:12px;color:var(--ac);text-decoration:none;border:1px solid var(--bd);background:#fff;border-radius:6px;padding:3px 9px}
.indice a:hover{background:#EEF4FA}
details.secao{background:var(--card);border:1px solid var(--bd);border-radius:10px;margin:12px 0;padding:0 16px}
details.secao>summary{cursor:pointer;font-weight:600;font-size:16px;padding:12px 0}
.bloco{margin:6px 0 22px}
.bloco h4{margin:0 0 6px;font-size:14px;color:#374151}
.nota{color:var(--sub);font-size:12px;margin:4px 0 0}
.vega,.svg,.tabela-wrap{overflow-x:auto;max-width:100%}
.tabela{border-collapse:collapse;font-size:12.5px;margin:0}
.tabela th,.tabela td{border-bottom:1px solid var(--bd);padding:4px 10px;text-align:right;white-space:nowrap}
.tabela thead th{background:#F3F4F6;position:sticky;top:0}
.tabela tbody th{text-align:left;background:#FAFAFA}
.aviso{background:var(--av);border-left:4px solid var(--avb);padding:8px 12px;border-radius:4px;font-size:13px;margin:8px 0}
.info{color:#374151;font-size:13px;margin:6px 0}
footer{color:var(--sub);font-size:12px;text-align:center;padding:20px}
@media print{.abas{display:none}details.secao{break-inside:avoid}header{background:#fff;color:#000}}
"""

_JS = """
(function () {
  var opcoes = {actions: {export: true, source: false, compiled: false, editor: false}, renderer: "canvas"};
  function desenhar(el) {
    if (el.dataset.ok) return;
    el.dataset.ok = "1";
    if (typeof vegaEmbed === "undefined") {
      el.innerHTML = '<p class="aviso">Não consegui carregar as bibliotecas Vega (sem internet?). Gere o relatório com RELATORIO_ESTATICO = True.</p>';
      return;
    }
    var spec = JSON.parse(document.getElementById(el.dataset.spec).textContent);
    vegaEmbed(el, spec, opcoes).catch(function (e) {
      el.innerHTML = '<p class="aviso">Erro ao desenhar o gráfico: ' + e + '</p>';
    });
  }
  var obs = new IntersectionObserver(function (entradas) {
    entradas.forEach(function (e) { if (e.isIntersecting) { desenhar(e.target); obs.unobserve(e.target); } });
  }, {rootMargin: "600px"});
  document.querySelectorAll(".vega").forEach(function (el) { obs.observe(el); });

  var abas = Array.prototype.slice.call(document.querySelectorAll(".aba"));
  var secs = Array.prototype.slice.call(document.querySelectorAll(".cenario"));
  function ir(i) {
    abas.forEach(function (a, j) { a.classList.toggle("ativa", i === j); });
    secs.forEach(function (s, j) { s.hidden = !(i === j || i === secs.length); });
    if (window.history && history.replaceState) history.replaceState(null, "", "#aba-" + i);
  }
  abas.forEach(function (a, i) { a.addEventListener("click", function () { ir(i); window.scrollTo(0, 0); }); });
  var m = /^#aba-(\\d+)$/.exec(location.hash);
  ir(m && +m[1] <= secs.length ? +m[1] : 0);
})();
"""


def _svg(spec):
    import vl_convert as vlc
    return vlc.vegalite_to_svg(json.dumps(spec, default=str))


def _cdn_vega(specs):
    """Scripts do Vega no CDN, na versão do Vega-Lite usada pelo Altair instalado."""
    major = 5
    for s in specs:
        achou = [p for p in str(s.get("$schema", "")).split("/") if p.startswith("v") and p[1:2].isdigit()]
        if achou:
            major = int(achou[-1][1:].split(".")[0])
            break
    vega, embed = (6, 7) if major >= 6 else (5, 6)
    base_url = "https://cdn.jsdelivr.net/npm"
    return (f'<script src="{base_url}/vega@{vega}"></script>\n'
            f'<script src="{base_url}/vega-lite@{major}"></script>\n'
            f'<script src="{base_url}/vega-embed@{embed}"></script>')


def gerar_relatorio_html(caminho, titulo, estatico=False):
    esc = _html.escape
    if not RELATORIO:
        print("Nada foi registrado ainda: rode as etapas anteriores.")
        return None

    cenarios_ord = []
    for it in RELATORIO:
        if it["cenario"] not in cenarios_ord:
            cenarios_ord.append(it["cenario"])
    cenarios_ord.sort(key=lambda c: not c.startswith("Comparação"))   # comparação em primeiro (ordem estável)

    specs_json, specs_raw, avisos = [], [], []
    secoes_html = []
    for ci, cen in enumerate(cenarios_ord):
        itens = [it for it in RELATORIO if it["cenario"] == cen]
        nomes_secao = []
        for it in itens:
            if it["secao"] not in nomes_secao:
                nomes_secao.append(it["secao"])
        indice = "".join(f'<a href="#s{ci}-{si}">{esc(s)}</a>' for si, s in enumerate(nomes_secao))
        partes = []
        for si, s in enumerate(nomes_secao):
            blocos = []
            for it in itens:
                if it["secao"] != s:
                    continue
                nota = f'<p class="nota">{esc(it["nota"])}</p>' if it.get("nota") else ""
                titulo_it = f'<h4>{esc(it["titulo"])}</h4>' if it.get("titulo") else ""
                if it["tipo"] == "grafico":
                    conteudo = None
                    if estatico:
                        try:
                            conteudo = f'<div class="svg">{_svg(it["spec"])}</div>'
                        except Exception as e:
                            avisos.append(f"SVG falhou em '{it['titulo']}': {e}")
                    if conteudo is None:
                        sid = len(specs_json)
                        specs_raw.append(it["spec"])
                        specs_json.append(json.dumps(it["spec"], default=str).replace("</", "<\\/"))
                        conteudo = f'<div class="vega" data-spec="spec-{sid}"></div>'
                    blocos.append(f'<div class="bloco">{titulo_it}{conteudo}{nota}</div>')
                elif it["tipo"] == "tabela":
                    blocos.append(f'<div class="bloco">{titulo_it}<div class="tabela-wrap">{it["html"]}</div>{nota}</div>')
                else:
                    classe = "aviso" if it.get("nivel") == "aviso" else "info"
                    blocos.append(f'<p class="{classe}">{esc(it["texto"])}</p>')
            partes.append(f'<details class="secao" id="s{ci}-{si}" open><summary>{esc(s)}</summary>{"".join(blocos)}</details>')
        secoes_html.append(f'<section class="cenario" id="aba{ci}"><h2 class="cen">{esc(cen)}</h2>'
                           f'<div class="indice">{indice}</div>{"".join(partes)}</section>')

    abas = "".join(f'<button class="aba" type="button">{esc(c)}</button>' for c in cenarios_ord)
    abas += '<button class="aba" type="button">Todos</button>'
    n_graf = sum(1 for it in RELATORIO if it["tipo"] == "grafico")
    n_tab = sum(1 for it in RELATORIO if it["tipo"] == "tabela")
    agora = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")
    meta = (f"Gerado em {agora} · base: {esc(str(CAMINHO_ARQUIVO))} · {len(cenarios_ord)} aba(s) · "
            f"{n_graf} gráficos · {n_tab} tabelas")
    scripts_spec = "\n".join(f'<script type="application/json" id="spec-{i}">{s}</script>' for i, s in enumerate(specs_json))
    vega_js = _cdn_vega(specs_raw) if specs_json else ""

    doc = (f'<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n'
           f'<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>{esc(titulo)}</title>\n'
           f'<style>{_CSS}</style>\n{vega_js}\n</head>\n<body>\n'
           f'<header><h1>{esc(titulo)}</h1><p>{meta}</p></header>\n'
           f'<nav class="abas">{abas}</nav>\n<main>\n{"".join(secoes_html)}\n</main>\n'
           f'<footer>Relatório gerado pelo notebook de quedas bruscas.</footer>\n'
           f'{scripts_spec}\n<script>{_JS}</script>\n</body>\n</html>\n')

    saida = Path(caminho)
    saida.write_text(doc, encoding="utf-8")
    for a in avisos[:5]:
        print("⚠", a)
    print(f"Relatório salvo em: {saida.resolve()}  ({saida.stat().st_size / 1e6:.1f} MB, {n_graf} gráficos, {n_tab} tabelas)")
    return saida


if GERAR_RELATORIO:
    caminho_relatorio = gerar_relatorio_html(ARQUIVO_RELATORIO, TITULO_RELATORIO, RELATORIO_ESTATICO)

Relatório salvo em: /content/relatorio_quedas_bruscas.html  (4.0 MB, 56 gráficos, 67 tabelas)
